# Attacker size study — straightforward A100 version

**Question:** Do larger fresh attackers recover more plaintext, and does more training change that?

Choose **A100** in Colab and run the notebook from top to bottom.

The default experiment is **1 Alice–Bob pair × 4 attacker architectures × 5 sizes × 3 attacker seeds = 60 attackers**. To repeat across three learned ciphers, change `PAIR_SEEDS = (7,)` to `PAIR_SEEDS = (7, 8, 9)`.

What happens:
1. Train CNN Alice and CNN Bob against the training **MLP Eve** for 5,000 steps.
2. Freeze that pair.
3. Train each fresh CNN, MLP, BiLSTM, and Transformer attacker at five sizes.
4. Repeat each attacker configuration with three initializations. Each attacker trains continuously to 20,000 steps, with test measurements at 5,000 and 20,000 steps.

A **pair seed** initializes Alice, Bob, and the training Eve. An **attacker seed** initializes a fresh attacker; it does not reset Alice or Bob.

Each optimizer update uses a fixed batch of **16,384 examples**, as in Week 2. Attackers run **one at a time**. BF16 and fused Adam provide GPU acceleration. There are no speed trials, automatic batch searches, gradient-accumulation routines, or concurrent CUDA streams.

All results are printed and plots use Seaborn inline. The notebook writes no checkpoints, CSVs, images, or other experiment files. Results/models exist in memory only. This notebook removes tuning overhead; it does not guarantee that all 60 training runs finish quickly.

## 1. Settings

These are the experiment controls. Keep them fixed during a run. `BATCH_SIZE` is the number of examples per weight update; `ATTACK_BUDGETS` are the update counts at which we measure test accuracy.

The number of steps stays the same for every size. Equal steps and batches mean equal examples, **not equal computation time**. The GPU memory figure is storage occupied by tensors, not a measure of GPU compute utilization.

In [ ]:
import json
import math
import time

import numpy as np
import pandas as pd
import seaborn as sns
import seaborn.objects as so
import torch
from torch import nn

PAIR_SEEDS = (7,)                       # Use (7, 8, 9) for the full 180-attacker study.
ATTACKER_SEEDS = (101, 102, 103)        # Three different initializations of each attacker.
FAMILY_NAMES = ("CNN", "MLP", "BiLSTM", "Transformer")
TARGET_SIZES = (5_000, 10_000, 20_000, 40_000, 80_000)

PAIR_STEPS = 5_000
ATTACK_BUDGETS = (5_000, 20_000)
BATCH_SIZE = 16_384
EVE_UPDATES = 2
AB_LR = 8e-4
ATTACK_LR = 1e-3
PRIVACY_WEIGHT = 8.0

VALIDATION_SIZE = 16_384
TEST_SIZE = 65_536
EVAL_BATCH_SIZE = 2_048               # Only evaluation is split to limit temporary memory.
LOG_EVERY = 1_000
BITS = 32

assert torch.cuda.is_available(), "Select an A100 GPU runtime in Colab."
assert torch.cuda.is_bf16_supported(), "A BF16-capable CUDA GPU is required."
device = torch.device("cuda")
gpu = torch.cuda.get_device_properties(0)
print(f"GPU: {gpu.name}; memory: {gpu.total_memory / 2**30:.1f} GiB; PyTorch: {torch.__version__}")
if "A100" not in gpu.name:
    print("These fixed-batch settings target A100. Select A100 before starting the study.")
torch.backends.cuda.matmul.allow_tf32 = True
torch.backends.cudnn.allow_tf32 = True
torch.backends.cudnn.benchmark = False
torch.backends.mha.set_fastpath_enabled(False)
torch.set_float32_matmul_precision("high")
sns.set_theme(style="whitegrid", context="notebook", palette="colorblind")

assert PAIR_SEEDS and len(set(PAIR_SEEDS)) == len(PAIR_SEEDS)
assert ATTACKER_SEEDS and len(set(ATTACKER_SEEDS)) == len(ATTACKER_SEEDS)
assert len(TARGET_SIZES) >= 2 and len(set(TARGET_SIZES)) == len(TARGET_SIZES)
assert ATTACK_BUDGETS == tuple(sorted(set(ATTACK_BUDGETS)))
assert all(n > 0 for n in (*TARGET_SIZES, *ATTACK_BUDGETS, PAIR_STEPS, BATCH_SIZE,
                           VALIDATION_SIZE, TEST_SIZE, EVAL_BATCH_SIZE, LOG_EVERY, EVE_UPDATES))
TOTAL_ATTACKERS = len(PAIR_SEEDS) * len(FAMILY_NAMES) * len(TARGET_SIZES) * len(ATTACKER_SEEDS)
CONFIG = dict(pair_seeds=PAIR_SEEDS, attacker_seeds=ATTACKER_SEEDS, families=FAMILY_NAMES,
              target_sizes=TARGET_SIZES, pair_steps=PAIR_STEPS, attack_budgets=ATTACK_BUDGETS,
              batch_size=BATCH_SIZE, eve_updates=EVE_UPDATES, ab_lr=AB_LR,
              attack_lr=ATTACK_LR, privacy_weight=PRIVACY_WEIGHT, bits=BITS,
              validation_size=VALIDATION_SIZE, test_size=TEST_SIZE,
              training_eve="MLP(32-133-218-32)")
print(f"{TOTAL_ATTACKERS} attackers; {TOTAL_ATTACKERS * len(ATTACK_BUDGETS)} test records.")
print("CONFIG_JSON " + json.dumps(CONFIG), flush=True)

## 2. Models and small helpers

Alice and Bob each have 40,609 parameters. The training MLP Eve has exactly the same count. Fresh attackers have the Week 2 architectures: only width changes within each family.

The helpers generate random examples, calculate the training loss, and measure recovery. **They do not run experiments when this cell is executed.** The final training cell is where training starts.

In [ ]:
class CryptCNN(nn.Module):
    """The CNN architecture used separately for Alice and Bob."""
    def __init__(self):
        """Build the 40,609-parameter network with 64 inputs and 32 outputs."""
        super().__init__()
        self.mix = nn.Linear(64, 192)
        self.conv1 = nn.Conv1d(1, 48, 5, padding="same")
        self.conv2 = nn.Conv1d(48, 96, 4, stride=2, padding=1)
        self.conv3 = nn.Conv1d(96, 48, 2, stride=3)
        self.conv4 = nn.Conv1d(48, 1, 1)

    def forward(self, x):
        """Map (message, key) or (ciphertext, key), shape (N, 64), to (N, 32)."""
        x = self.mix(x).unsqueeze(1)
        x = torch.sigmoid(self.conv1(x))
        x = torch.sigmoid(self.conv2(x))
        x = torch.sigmoid(self.conv3(x))
        return torch.tanh(self.conv4(x).squeeze(1))


class MLPEve(nn.Module):
    """Two-hidden-layer MLP receiving ciphertext alone."""
    def __init__(self, width, hidden2=None):
        """Use equal hidden widths by default; Stage 1 uses widths 133 and 218."""
        super().__init__()
        hidden2 = width if hidden2 is None else hidden2
        self.net = nn.Sequential(nn.Linear(32, width), nn.ReLU(),
                                 nn.Linear(width, hidden2), nn.ReLU(),
                                 nn.Linear(hidden2, 32), nn.Tanh())

    def forward(self, ciphertext):
        """Return (N, 32) plaintext predictions from (N, 32) ciphertexts."""
        return self.net(ciphertext)


class CNNEve(nn.Module):
    """Fresh CNN attacker with width-controlled channel counts."""
    def __init__(self, width):
        """Build the ciphertext-only CNN; width changes its parameter count."""
        super().__init__()
        self.mix = nn.Linear(32, 128)
        self.conv1 = nn.Conv1d(1, width, 5, padding="same")
        self.conv2 = nn.Conv1d(width, 2 * width, 4, stride=2, padding=1)
        self.conv3 = nn.Conv1d(2 * width, width, 2, stride=2)
        self.conv4 = nn.Conv1d(width, 1, 1)

    def forward(self, ciphertext):
        """Return (N, 32) plaintext predictions using no key."""
        x = self.mix(ciphertext).unsqueeze(1)
        x = torch.sigmoid(self.conv1(x))
        x = torch.sigmoid(self.conv2(x))
        x = torch.sigmoid(self.conv3(x))
        return torch.tanh(self.conv4(x).squeeze(1))


class BiLSTMEve(nn.Module):
    """Fresh attacker that reads ciphertext positions in both directions."""
    def __init__(self, width):
        """Set each LSTM direction's hidden size to width."""
        super().__init__()
        self.lstm = nn.LSTM(1, width, batch_first=True, bidirectional=True)
        self.readout = nn.Linear(2 * width, 1)

    def forward(self, ciphertext):
        """Treat the 32 ciphertext values as a sequence and predict 32 bits."""
        sequence, _ = self.lstm(ciphertext.unsqueeze(-1))
        return torch.tanh(self.readout(sequence).squeeze(-1))


class TransformerEve(nn.Module):
    """Fresh two-layer, four-head Transformer attacker with no dropout."""
    def __init__(self, width):
        """Set the embedding width; it must be divisible by four."""
        super().__init__()
        self.input_projection = nn.Linear(1, width)
        self.position = nn.Parameter(torch.zeros(1, 32, width))
        layer = nn.TransformerEncoderLayer(width, 4, 2 * width, dropout=0.0,
                                            batch_first=True, activation="gelu")
        self.encoder = nn.TransformerEncoder(layer, num_layers=2, enable_nested_tensor=False)
        self.readout = nn.Linear(width, 1)

    def forward(self, ciphertext):
        """Attend to ciphertext positions and return (N, 32) plaintext predictions."""
        x = self.input_projection(ciphertext.unsqueeze(-1)) + self.position
        return torch.tanh(self.readout(self.encoder(x)).squeeze(-1))


FAMILIES = dict(CNN=CNNEve, MLP=MLPEve, BiLSTM=BiLSTMEve, Transformer=TransformerEve)

def parameter_count(model):
    """Return the number of parameter values, including any frozen weights."""
    return sum(p.numel() for p in model.parameters())

def choose_width(family, target):
    """Return the width nearest target parameters, using formulas rather than GPU trials."""
    formulas = dict(CNN=lambda w: 12*w*w + 10*w + 4225,
                    MLP=lambda w: w*w + 66*w + 32,
                    BiLSTM=lambda w: 8*w*w + 26*w + 1,
                    Transformer=lambda w: 16*w*w + 57*w + 1)
    candidates = range(8, 321, 4) if family == "Transformer" else range(8, 321)
    return min(candidates, key=lambda w: abs(formulas[family](w) - target))

def amp():
    """Use BF16 for eligible GPU operations; weights and loss accumulation stay float32."""
    return torch.autocast(device_type="cuda", dtype=torch.bfloat16)

def random_batch(n, rng):
    """Return plaintexts and keys of shape (n, 32), with random -1/+1 entries."""
    message = torch.randint(0, 2, (n, BITS), generator=rng, device=device).float() * 2 - 1
    key = torch.randint(0, 2, (n, BITS), generator=rng, device=device).float() * 2 - 1
    return message, key

def reconstruction_loss(prediction, message):
    """Return mean continuous L1 error per message: 0 is perfect, zero predictions give 16."""
    return ((prediction.float() - message).abs() / 2).sum(dim=1).mean()

@torch.no_grad()
def evaluate(model, inputs, message):
    """Measure a model without training it, processing EVAL_BATCH_SIZE examples at a time.

    Returns bit accuracy, mean wrong bits per message, full-message accuracy,
    continuous L1 error, and a standard error computed across messages.
    Inputs are ciphertexts for an attacker or ciphertext/key concatenations for Bob.
    The caller puts the model in eval mode. Nonfinite predictions raise an error.
    """
    totals = torch.zeros(5, device=device, dtype=torch.float64)
    for start in range(0, len(message), EVAL_BATCH_SIZE):
        labels = message[start:start+EVAL_BATCH_SIZE]
        with amp():
            prediction = model(inputs[start:start+EVAL_BATCH_SIZE])
        errors = ((prediction >= 0) != (labels >= 0)).float().sum(dim=1)
        continuous = ((prediction.float() - labels).abs() / 2).sum(dim=1)
        totals += torch.stack((errors.sum(), errors.square().sum(),
                               (errors == 0).sum(), continuous.sum(),
                               torch.isfinite(prediction).all().double())).double()
    errors, errors_squared, full, l1, finite = totals.cpu().tolist()
    n = len(message)
    if finite != math.ceil(n / EVAL_BATCH_SIZE):
        raise FloatingPointError("Nonfinite evaluation predictions.")
    variance = max(0.0, (errors_squared - errors*errors/n) / max(n-1, 1))
    return dict(bit_accuracy=1-errors/(BITS*n), hamming_error=errors/n,
                full_message_accuracy=full/n, reconstruction_l1=l1/n,
                bit_accuracy_se=math.sqrt(variance/n)/BITS, eval_messages=n)

SIZE_PLAN = {(family, target): choose_width(family, target)
             for family in FAMILY_NAMES for target in TARGET_SIZES}
assert parameter_count(CryptCNN()) == parameter_count(MLPEve(133, 218)) == 40_609
print(pd.DataFrame([
    dict(family=family, target_params=target, width=width,
         actual_params=parameter_count(FAMILIES[family](width)))
    for (family, target), width in SIZE_PLAN.items()
]).to_string(index=False))

## 3. How Alice and Bob learn

`train_pair(seed)` does the entire first stage. Each step:
1. Update training MLP Eve twice using ciphertext from Alice.
2. Update Alice and Bob once, encouraging reliable communication and Eve error near chance.

After the configured steps, all three networks are frozen. Bob is measured to check communication reliability. Separate validation and test examples are prepared for the fresh attackers. These held-out examples are never used for weight updates.

This cell only defines the function; training starts in section 6.

In [ ]:
def train_pair(pair_seed):
    """Train one Alice-Bob pair against MLP Eve, then return frozen Alice and held-out data.

    pair_seed initializes all three Stage-1 models. Separate sample seeds control
    training, validation, and test examples. Returns a dictionary with seed, alice,
    validation=(ciphertext, plaintext), test=(ciphertext, plaintext), and metrics.
    Prints progress and PAIR_JSON; saves no files.
    """
    torch.manual_seed(pair_seed)
    torch.cuda.manual_seed_all(pair_seed)
    alice = CryptCNN().to(device).train()
    bob = CryptCNN().to(device).train()
    training_eve = MLPEve(133, 218).to(device).train()
    ab_optimizer = torch.optim.Adam(list(alice.parameters()) + list(bob.parameters()),
                                    lr=AB_LR, fused=True)
    eve_optimizer = torch.optim.Adam(training_eve.parameters(), lr=ATTACK_LR, fused=True)
    rng = torch.Generator(device=device).manual_seed(100_000 + pair_seed)
    torch.cuda.synchronize()
    started = time.perf_counter()
    print(f"TRAIN PAIR seed={pair_seed}: {PAIR_STEPS:,} steps against MLP Eve", flush=True)

    for step in range(1, PAIR_STEPS + 1):
        # Eve learns to recover the plaintext from Alice's ciphertext.
        for _ in range(EVE_UPDATES):
            message, key = random_batch(BATCH_SIZE, rng)
            with torch.no_grad(), amp():
                ciphertext = alice(torch.cat((message, key), dim=1))
            with amp():
                eve_loss = reconstruction_loss(training_eve(ciphertext), message)
            eve_optimizer.zero_grad(set_to_none=True)
            eve_loss.backward()
            eve_optimizer.step()

        # Alice and Bob learn to communicate while making Eve's error approach 16.
        message, key = random_batch(BATCH_SIZE, rng)
        training_eve.requires_grad_(False)
        with amp():
            ciphertext = alice(torch.cat((message, key), dim=1))
            bob_loss = reconstruction_loss(bob(torch.cat((ciphertext, key), dim=1)), message)
            eve_error = reconstruction_loss(training_eve(ciphertext), message)
            privacy_loss = ((BITS / 2 - eve_error) / (BITS / 2)).square()
            total_loss = bob_loss + PRIVACY_WEIGHT * privacy_loss
        ab_optimizer.zero_grad(set_to_none=True)
        total_loss.backward()
        ab_optimizer.step()
        training_eve.requires_grad_(True)

        if step == 1 or step % LOG_EVERY == 0 or step == PAIR_STEPS:
            torch.cuda.synchronize()
            bob_value, eve_value = bob_loss.item(), eve_error.item()
            if not all(math.isfinite(v) for v in (bob_value, eve_value)):
                raise FloatingPointError("Nonfinite Stage-1 loss.")
            print(f"PAIR {pair_seed} step={step:,}/{PAIR_STEPS:,} "
                  f"Bob L1={bob_value:.3f} Eve L1={eve_value:.3f} "
                  f"elapsed={(time.perf_counter()-started)/60:.1f} min", flush=True)

    alice.eval().requires_grad_(False)
    bob.eval().requires_grad_(False)
    training_eve.eval().requires_grad_(False)
    held_out = {}
    bob_scores = {}
    for name, size, sample_seed in (("validation", VALIDATION_SIZE, 200_000 + pair_seed),
                                    ("test", TEST_SIZE, 300_000 + pair_seed)):
        rng = torch.Generator(device=device).manual_seed(sample_seed)
        message, key = random_batch(size, rng)
        ciphertext_chunks = []
        with torch.no_grad(), amp():
            for start in range(0, size, EVAL_BATCH_SIZE):
                ciphertext_chunks.append(alice(torch.cat(
                    (message[start:start+EVAL_BATCH_SIZE], key[start:start+EVAL_BATCH_SIZE]), dim=1)))
        ciphertext = torch.cat(ciphertext_chunks)
        held_out[name] = (ciphertext, message)
        bob_scores[name] = evaluate(bob, torch.cat((ciphertext, key), dim=1), message)

    row = dict(pair_seed=pair_seed, training_eve="MLP(32-133-218-32)", params_each=40_609,
               pair_steps=PAIR_STEPS, batch_size=BATCH_SIZE,
               bob_validation=bob_scores["validation"], bob_test=bob_scores["test"],
               training_eve_test=evaluate(training_eve, *held_out["test"]),
               reliable_bob=bob_scores["test"]["bit_accuracy"] >= 0.99)
    print("PAIR_JSON " + json.dumps(row, allow_nan=False), flush=True)
    if not row["reliable_bob"]:
        print("Bob is below 99% bit accuracy. Keep this flag when interpreting the attacks.", flush=True)
    return dict(seed=pair_seed, alice=alice, validation=held_out["validation"],
                test=held_out["test"], metrics=row)

## 4. How one fresh attacker learns

`train_attacker(pair, family, target, attacker_seed)` trains **one** attacker from scratch.

For example, `train_attacker(pair, "MLP", 20_000, 101)` means: attack this frozen pair using a roughly 20k-parameter MLP initialized with seed 101.

Each step generates 16,384 new examples, encrypts them with frozen Alice, and updates the attacker **once**. Every size and initialization sees the same sequence of training examples for that pair. Only the attacker's initialization changes across seeds 101, 102, and 103.

Validation measures learning every 1,000 steps. The held-out test set is used only at 5,000 and 20,000 steps; the same attacker continues training between those measurements.

In [ ]:
def train_attacker(pair, family, target, attacker_seed):
    """Train one fresh attacker to max(ATTACK_BUDGETS), recording both budgets.

    pair contains frozen Alice and shared held-out sets from train_pair().
    family chooses the architecture; target selects its nearest parameter size.
    attacker_seed initializes only this attacker, never Alice/Bob.

    Appends learning curves to history and test measurements to results, printing
    each RESULT_JSON immediately. The function returns None and saves no files.
    """
    width = SIZE_PLAN[family, target]
    torch.manual_seed(attacker_seed)
    torch.cuda.manual_seed_all(attacker_seed)
    attacker = FAMILIES[family](width).to(device).train()
    optimizer = torch.optim.Adam(attacker.parameters(), lr=ATTACK_LR, fused=True)
    rng = torch.Generator(device=device).manual_seed(400_000 + pair["seed"])
    actual_params = parameter_count(attacker)
    best_validation_l1 = float("inf")
    torch.cuda.reset_peak_memory_stats()
    torch.cuda.synchronize()
    started = time.perf_counter()
    print(f"START pair={pair['seed']} {family} target={target:,} actual={actual_params:,} "
          f"attacker_seed={attacker_seed}", flush=True)

    for step in range(1, max(ATTACK_BUDGETS) + 1):
        message, key = random_batch(BATCH_SIZE, rng)
        with torch.no_grad(), amp():
            ciphertext = pair["alice"](torch.cat((message, key), dim=1))
        with amp():
            loss = reconstruction_loss(attacker(ciphertext), message)
        optimizer.zero_grad(set_to_none=True)
        loss.backward()
        optimizer.step()

        if step == 1 or step % LOG_EVERY == 0 or step in ATTACK_BUDGETS:
            train_l1 = loss.item()
            if not math.isfinite(train_l1):
                raise FloatingPointError("Nonfinite attacker loss.")
            attacker.eval()
            validation = evaluate(attacker, *pair["validation"])
            best_validation_l1 = min(best_validation_l1, validation["reconstruction_l1"])
            trace = dict(pair_seed=pair["seed"], family=family, target_params=target,
                         actual_params=actual_params, width=width, attacker_seed=attacker_seed,
                         step=step, train_l1=train_l1,
                         validation_bit_accuracy=validation["bit_accuracy"],
                         validation_l1=validation["reconstruction_l1"],
                         best_validation_l1=best_validation_l1)
            history.append(trace)

            if step in ATTACK_BUDGETS:
                test_scores = evaluate(attacker, *pair["test"])
                row = dict(**trace, **test_scores, batch_size=BATCH_SIZE,
                           training_examples=step*BATCH_SIZE, attack_lr=ATTACK_LR,
                           training_eve="MLP(32-133-218-32)",
                           data_seed=400_000+pair["seed"],
                           bob_bit_accuracy=pair["metrics"]["bob_test"]["bit_accuracy"],
                           reliable_bob=pair["metrics"]["reliable_bob"])
                results.append(row)
                print("RESULT_JSON " + json.dumps(row, allow_nan=False), flush=True)

            torch.cuda.synchronize()
            elapsed = time.perf_counter() - started
            remaining = elapsed / step * (max(ATTACK_BUDGETS)-step)
            print(f"{family} size={actual_params:,} seed={attacker_seed} "
                  f"step={step:,}/{max(ATTACK_BUDGETS):,} "
                  f"validation={validation['bit_accuracy']:.2%} "
                  f"elapsed={elapsed/60:.1f} min remaining~{remaining/60:.1f} min "
                  f"peak GPU={torch.cuda.max_memory_allocated()/2**30:.1f} GiB", flush=True)
            attacker.train()

    # The trained attacker is no longer needed once its final metrics are recorded.
    del attacker, optimizer
    torch.cuda.empty_cache()

## 5. Displaying the results

`show_results()` prints the current table and draws the size curves. Thin lines show the individual attacker seeds. Thick lines average attacker seeds within each pair, then average across pairs. With several pairs, the band is one standard deviation **between pair means**.

Zero on the plots means 50% bit accuracy; +3 means 53%. The table gives accuracy as a percentage. `show_learning_curves()` displays validation progress.

The interpretation stays focused on size: do larger attackers consistently do better within the same architecture? Longer training, initialization variability, and differences between learned ciphers can affect that answer. A one-pair run cannot establish that a trend generalizes to other ciphers.

In [ ]:
def show_results():
    """Print current test results and display individual/mean size curves with Seaborn.

    Uses the results list without changing it. Average attacker seeds within each
    pair first; only independent pair means contribute to the between-pair band.
    Incomplete results are allowed and labeled with their available record count.
    """
    if not results:
        print("No test measurements yet.")
        return
    df = pd.DataFrame(results)
    expected = TOTAL_ATTACKERS * len(ATTACK_BUDGETS)
    print(f"Test records available: {len(df)}/{expected}")
    keys = ["pair_seed", "family", "target_params", "actual_params", "step"]
    pair_means = df.groupby(keys, as_index=False).agg(
        bit_accuracy=("bit_accuracy", "mean"),
        initialization_sd=("bit_accuracy", "std"),
        attacker_seeds=("attacker_seed", "nunique"),
        full_message_accuracy=("full_message_accuracy", "mean"),
        hamming_error=("hamming_error", "mean"),
    )
    table = pair_means.copy()
    table["bit_accuracy_percent"] = table.pop("bit_accuracy") * 100
    table["initialization_sd_pp"] = table.pop("initialization_sd") * 100
    print(table.to_string(index=False, float_format=lambda x: f"{x:.4f}"))

    df["advantage_pp"] = (df["bit_accuracy"] - 0.5) * 100
    df["budget"] = df["step"].map(lambda n: f"{n:,} updates")
    df["trial"] = df["pair_seed"].astype(str) + "/" + df["attacker_seed"].astype(str)
    (so.Plot(df, x="actual_params", y="advantage_pp", color="budget")
       .facet(col="family", wrap=2)
       .add(so.Line(alpha=.45), group="trial")
       .add(so.Dot(alpha=.6, pointsize=3))
       .scale(x="log")
       .label(x="Actual attacker parameters", y="Bit accuracy above 50% (pp)",
              title="Individual attacker initializations")
       .layout(size=(12, 8))
       .show())

    pair_means["advantage_pp"] = (pair_means["bit_accuracy"] - 0.5) * 100
    pair_means["budget"] = pair_means["step"].map(lambda n: f"{n:,} updates")
    plot = so.Plot(pair_means, x="actual_params", y="advantage_pp", color="budget").facet(col="family", wrap=2)
    if pair_means["pair_seed"].nunique() > 1:
        plot = plot.add(so.Band(alpha=.2), so.Est(errorbar="sd"))
    (plot.add(so.Line(linewidth=2), so.Agg())
         .add(so.Dot(pointsize=5), so.Agg())
         .scale(x="log")
         .label(x="Actual attacker parameters", y="Mean bit accuracy above 50% (pp)",
                title="Mean recovery; band, if present, is between-pair SD")
         .layout(size=(12, 8))
         .show())


def show_learning_curves():
    """Plot validation accuracy and L1 loss over time, averaging attacker seeds."""
    if not history:
        return
    df = pd.DataFrame(history)
    df["capacity"] = df["target_params"].map(lambda n: f"{n:,} parameters")
    df["pair"] = df["pair_seed"].map(lambda n: f"Pair {n}")
    df["validation_percent"] = df["validation_bit_accuracy"] * 100
    for metric, label in (("validation_percent", "Validation bit accuracy (%)"),
                          ("validation_l1", "Validation continuous L1 error")):
        (so.Plot(df, x="step", y=metric, color="capacity")
           .facet(col="family", row="pair")
           .add(so.Line(), so.Agg())
           .label(x="Attacker updates", y=label, title="Validation learning curves")
           .layout(size=(16, 3.5 * df["pair"].nunique()))
           .show())

## 6. Run the experiment

**This is the cell that starts training.** Read the loops: for each pair, train it once; for each attacker architecture and size, train three fresh initializations.

Run this cell once. It prints progress for each model and test results as soon as each budget finishes. Plots update after each architecture. **Rerunning this cell starts the study from scratch and clears its in-memory result lists.** If interrupted, run section 7 to inspect already recorded results instead. A runtime reset loses the models and in-memory lists.

For a quick execution check before a long run, temporarily set `PAIR_STEPS = 2`, `ATTACK_BUDGETS = (2, 4)`, `BATCH_SIZE = 64`, `VALIDATION_SIZE = 64`, `TEST_SIZE = 128`, and `LOG_EVERY = 2` in the settings cell. These short-run scores are not scientific results. Restore the settings and rerun setup before the real study.

In [ ]:
# Fresh lists make it explicit that running this cell starts a NEW experiment.
results = []
history = []
pair_results = []
finished_attackers = 0
study_started = time.perf_counter()
print("STARTING NEW STUDY", flush=True)

for pair_seed in PAIR_SEEDS:
    pair = train_pair(pair_seed)
    pair_results.append(pair["metrics"])

    for family in FAMILY_NAMES:
        for target in TARGET_SIZES:
            for attacker_seed in ATTACKER_SEEDS:
                train_attacker(pair, family, target, attacker_seed)
                finished_attackers += 1
                print(f"FINISHED {finished_attackers}/{TOTAL_ATTACKERS} attackers; "
                      f"study elapsed={(time.perf_counter()-study_started)/3600:.2f} hours",
                      flush=True)
        show_results()

    del pair
    torch.cuda.empty_cache()

print(f"STUDY FINISHED in {(time.perf_counter()-study_started)/3600:.2f} hours", flush=True)

## 7. Final comparison

This cell also works after an interruption as long as the runtime is alive.

The size-effect table uses only complete size sweeps with all attacker seeds. Its slope is the accuracy change, in percentage points, per doubling of actual parameters; a consistent positive slope across pairs would support a size effect. The budget-gain table compares the same models at the short and long budgets.

A secondary table selects the initialization with the lowest validation L1 at each budget and shows its test accuracy. Test scores do not choose the winner. There are no saved "best model" weights or test-based stopping rules.

In [ ]:
show_results()
show_learning_curves()

if results:
    df = pd.DataFrame(results)
    means = df.groupby(["pair_seed", "family", "target_params", "actual_params", "step"],
                      as_index=False).agg(bit_accuracy=("bit_accuracy", "mean"),
                                          seeds=("attacker_seed", "nunique"))
    effects = []
    for (pair_seed, family, step), group in means.groupby(["pair_seed", "family", "step"]):
        if set(group["target_params"]) != set(TARGET_SIZES) or not (group["seeds"] == len(ATTACKER_SEEDS)).all():
            continue
        group = group.sort_values("actual_params")
        accuracies = group["bit_accuracy"].to_numpy() * 100
        slope = np.polyfit(np.log2(group["actual_params"].to_numpy()), accuracies, 1)[0]
        effects.append(dict(pair_seed=int(pair_seed), family=family, step=int(step),
                            slope_pp_per_doubling=float(slope),
                            largest_minus_smallest_pp=float(accuracies[-1] - accuracies[0])))
    print("\nSIZE EFFECTS: averages over attacker initializations, reported separately for each pair.")
    print(pd.DataFrame(effects).to_string(index=False))
    for row in effects:
        print("SIZE_EFFECT_JSON " + json.dumps(row, allow_nan=False), flush=True)

    paired = means.pivot(index=["pair_seed", "family", "target_params"],
                         columns="step", values="bit_accuracy")
    short, long = min(ATTACK_BUDGETS), max(ATTACK_BUDGETS)
    if short in paired and long in paired:
        print("\nGAIN FROM EXTRA TRAINING: long minus short budget, in percentage points.")
        print(((paired[long]-paired[short])*100).dropna().rename("gain_pp").to_string())

    selected = df.loc[df.groupby(["pair_seed", "family", "target_params", "step"])["validation_l1"].idxmin()]
    print("\nVALIDATION-SELECTED INITIALIZATION: test accuracy was not used for selection.")
    print(selected[["pair_seed", "family", "target_params", "step", "attacker_seed",
                    "validation_l1", "bit_accuracy", "full_message_accuracy"]].to_string(index=False))
    print("\nBob reliability:", {row["pair_seed"]: row["reliable_bob"] for row in pair_results})

expected_records = TOTAL_ATTACKERS * len(ATTACK_BUDGETS)
print(f"Finished attackers: {finished_attackers}/{TOTAL_ATTACKERS}; "
      f"test records: {len(results)}/{expected_records}.")
print("STUDY_STATUS_JSON " + json.dumps(dict(
    complete=finished_attackers == TOTAL_ATTACKERS and len(results) == expected_records,
    finished_attackers=finished_attackers, expected_attackers=TOTAL_ATTACKERS,
    test_records=len(results), expected_records=expected_records)))